In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

os.environ['PYTHONHASHSEED'] = '42'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

I0000 00:00:1792990436.154522    3133 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1792990439.006098    3133 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
data = pd.read_csv('English2Hindi.csv', encoding='utf-8').dropna(subset=['English', 'Hindi'])
# Keep short sentence pairs (<= 10 words) and a 10k sample so padding and training stay manageable
eng_len = data['English'].str.split().str.len()
hin_len = data['Hindi'].str.split().str.len()
data = data[(eng_len <= 10) & (hin_len <= 10)].sample(n=10000, random_state=42).reset_index(drop=True)

X_data = data['English'].astype(str).str.lower().str.strip()
y_data = data['Hindi'].astype(str).str.strip().apply(lambda x: 'startseq ' + x + ' endseq')

eng_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
eng_tokenizer.fit_on_texts(X_data)
num_enc_toks = len(eng_tokenizer.word_index) + 1
hin_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
hin_tokenizer.fit_on_texts(y_data)
num_dec_toks = len(hin_tokenizer.word_index) + 1
enc_input_sequences = eng_tokenizer.texts_to_sequences(X_data)
dec_sequences = hin_tokenizer.texts_to_sequences(y_data)

max_len_src = max(len(seq) for seq in enc_input_sequences)
max_len_tar = max(len(seq) for seq in dec_sequences)

encoder_input_data = pad_sequences(enc_input_sequences, maxlen=max_len_src, padding='post')
padded_dec_sequences = pad_sequences(dec_sequences, maxlen=max_len_tar, padding='post')
decoder_input_data = padded_dec_sequences[:, :-1]
decoder_output_data = padded_dec_sequences[:, 1:]

max_len_tar = max_len_tar - 1
(encoder_input_train, encoder_input_test,
 decoder_input_train, decoder_input_test,
 decoder_output_train, decoder_output_test,
 X_train, X_test,
 y_train, y_test) = train_test_split(
    encoder_input_data, decoder_input_data, decoder_output_data, X_data, data['Hindi'],
    test_size=0.2, random_state=42
)

In [3]:
latent_dim = 250
enc_inps = Input(shape=(max_len_src,))
enc_emb = Embedding(num_enc_toks, latent_dim, mask_zero=True)(enc_inps)
enc_outputs, st_h, st_c = LSTM(latent_dim, return_state=True)(enc_emb)
enc_states = [st_h, st_c]

dec_inps = Input(shape=(max_len_tar,))
dec_emb = Embedding(num_dec_toks, latent_dim, mask_zero=True)(dec_inps)
dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_outputs, _, _ = dec_lstm(dec_emb, initial_state=enc_states)
dec_dense = Dense(num_dec_toks, activation='softmax')
dec_outputs = dec_dense(dec_outputs)

model = Model([enc_inps, dec_inps], dec_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

E0000 00:00:1792990443.118900    3133 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
print("Training model...")
model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_output_train,
    batch_size=16,
    epochs=50,
    validation_data=([encoder_input_test, decoder_input_test], decoder_output_test)
)

Training model...
Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 59s 113ms/step - accuracy: 0.2606 - loss: 5.9006 - val_accuracy: 0.2792 - val_loss: 5.5057
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 54s 108ms/step - accuracy: 0.2902 - loss: 5.0270 - val_accuracy: 0.2918 - val_loss: 5.3937
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 56s 112ms/step - accuracy: 0.3115 - loss: 4.5975 - val_accuracy: 0.3008 - val_loss: 5.4380
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 51s 102ms/step - accuracy: 0.3292 - loss: 4.2188 - val_accuracy: 0.3090 - val_loss: 5.4858
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 48s 96ms/step - accuracy: 0.3515 - loss: 3.8592 - val_accuracy: 0.3158 - val_loss: 5.5413
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 51s 102ms/step - accuracy: 0.3801 - loss: 3.5040 - val_accuracy: 0.3209 - val_loss: 5.5845
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 58s 117ms/step - accuracy: 0.4185 - loss: 3.1428 - val_accuracy: 0.3174 - val_loss: 5.6792
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 52s 103ms/step - accuracy:

In [5]:
enc_model = Model(enc_inps, enc_states)
dec_st_inp_h = Input(shape=(latent_dim,))
dec_st_inp_c = Input(shape=(latent_dim,))
dec_states_inps = [dec_st_inp_h, dec_st_inp_c]
dec_emb2 = model.layers[3](dec_inps)
dec_outputs2, st_h2, st_c2 = model.layers[5](dec_emb2, initial_state=dec_states_inps)
dec_outputs2 = model.layers[6](dec_outputs2)
dec_model = Model([dec_inps] + dec_states_inps, [dec_outputs2] + [st_h2, st_c2])
rev_tar_char_idx = {i: word for word, i in hin_tokenizer.word_index.items()}

In [6]:
def translate(input_text):
    seq = eng_tokenizer.texts_to_sequences([input_text])
    padded_seq = pad_sequences(seq, maxlen=max_len_src, padding='post')
    states_value = enc_model.predict(padded_seq, verbose=0)
    tar_seq = np.zeros((1, max_len_tar))
    tar_seq[0, 0] = hin_tokenizer.word_index['startseq']
    dec_sen = []
    for t in range(max_len_tar - 1):
        output_toks, h, c = dec_model.predict([tar_seq] + states_value, verbose=0)
        sampled_tok_idx = np.argmax(output_toks[0, t, :])
        if sampled_tok_idx == 0:
            break
        sampled_char = rev_tar_char_idx.get(sampled_tok_idx, '')
        if sampled_char == 'endseq':
            break
        dec_sen.append(sampled_char)
        tar_seq[0, t + 1] = sampled_tok_idx
    return ' '.join(dec_sen)

print("\n--- Translation Test (3 Sampled Unseen Test Sentences) ---")
sample_indices = X_test.sample(n=3, random_state=42).index
for i in sample_indices:
    test_phrase = X_test.loc[i]
    print(f"English Input: {test_phrase}")
    print(f"Predicted Hindi: {translate(test_phrase)}")
    print(f"Actual Expected Hindi: {y_test.loc[i]}")


--- Translation Test (3 Sampled Unseen Test Sentences) ---
English Input: [make a difference.]


Predicted Hindi: तीन तरह से
Actual Expected Hindi: [बदलाव लाने का ज़रिया बनिये ।]
English Input: today , that no longer holds .


Predicted Hindi: जो कि कई बार पर भी बहुत ही
Actual Expected Hindi: लेकिन आज स्थिति पूरी तरह से अलग है .
English Input: parashar and satyavathi's son


Predicted Hindi: और जन्म और स्थिति
Actual Expected Hindi: पाराशर और सत्यवती के पुत्र।
